### Import Required Libraries

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

### Handle Missing Values

In [22]:
def handle_missing(df):
    df['age'] = df['age'].fillna(df['age'].median())
    df['sex'] = df['sex'].replace('unknown', 'unspecified')
    return df

### Handle Age Outliers

In [23]:

def handle_outliers(df):

    Q1, Q3 = df['age'].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df['age'] = df['age'].clip(lower=lower, upper=upper)
    return df

### Train-Test Split by Lesion ID not by row to avoid (data leakage)

In [24]:
def split_data(df):
    unique_lesions = df[['lesion_id', 'dx']].drop_duplicates()

    train_lesions, test_lesions = train_test_split(
        unique_lesions,
        test_size=0.2,
        stratify=unique_lesions['dx'],
        random_state=42
    )

    train_df = df[df['lesion_id'].isin(train_lesions['lesion_id'])]
    test_df  = df[df['lesion_id'].isin(test_lesions['lesion_id'])]

    return train_df, test_df


### Compute Class Weights to solve target imbalance 

In [25]:
def get_class_weights(train_df):
    classes = np.unique(train_df['dx'])
    weights = compute_class_weight('balanced', classes=classes, y=train_df['dx'])
    class_weights = dict(zip(classes, weights))
    return class_weights

### Data Preprocessing Pipeline for CV

In [26]:
def run_pipeline_CV(filepath):
    print('1. Loading Data')
    df = pd.read_csv(filepath)
    print(f'Loaded: {df.shape}')

    print('\n 2. Handle Missing')
    df = handle_missing(df)
    print(f'Missing after: {df.isnull().sum().sum()}')

    print('\n 3. Handle Outliers ')
    df = handle_outliers(df)
    print(f'Age range after clip: {df["age"].min()} – {df["age"].max()}')

    print('\n 4. Split Data ')
    train_df, test_df = split_data(df)
    print(f'Train: {len(train_df)} | Test: {len(test_df)}')

    print('\n5. Save CV Files ')
    train_df[['image_id', 'dx']].to_csv('../data/cv_train.csv', index=False)
    test_df [['image_id', 'dx']].to_csv('../data/cv_test.csv',  index=False)
    print('Saved: cv_train.csv, cv_test.csv')

    print('\n=== 6. Leakage Check ===')
    overlap = set(train_df['lesion_id']) & set(test_df['lesion_id'])
    print(f'Shared lesions: {len(overlap)}') 

    return train_df, test_df

In [27]:
train_df, test_df = run_pipeline_CV('../data/HAM10000_metadata.csv')

1. Loading Data
Loaded: (10015, 7)

 2. Handle Missing
Missing after: 0

 3. Handle Outliers 
Age range after clip: 2.5 – 85.0

 4. Split Data 
Train: 8020 | Test: 1995

5. Save CV Files 
Saved: cv_train.csv, cv_test.csv

=== 6. Leakage Check ===
Shared lesions: 0


## pre_processing for NLP

### Build Text Column for NLP

In [28]:
def build_text(df):
    df['text'] = (
        'Dermatology case: ' + df['age'].astype(int).astype(str) +
        ' year old ' + df['sex'] + ' patient. ' +
        'Lesion location: ' + df['localization'] + '. ' +
        'Diagnosis method: ' + df['dx_type'] + '.'
    )
    return df


In [29]:
def map_risk(dx):
    low    = ['nv', 'bkl', 'df', 'vasc']
    medium = ['bcc', 'akiec']
    high   = ['mel']
    if dx in low:    return 'low'
    if dx in medium: return 'medium'
    if dx in high:   return 'high'

### Save NLP Files


In [30]:
def save_nlp_files(train_df, test_df):
    train_df[['text', 'risk']].to_csv('../data/nlp_train.csv', index=False)
    test_df [['text', 'risk']].to_csv('../data/nlp_test.csv',  index=False)
    print('Saved: nlp_train.csv, nlp_test.csv')

### Data Preprocessing Pipeline for NLP

In [31]:
def run_pipeline_NLP(filepath):
    print('1. Loading Data')
    df = pd.read_csv(filepath)
    print(f'Loaded: {df.shape}')

    print('\n2. Handle Missing')
    df = handle_missing(df)
    print(f'Missing after: {df.isnull().sum().sum()}')

    print('\n3. Handle Outliers')
    df = handle_outliers(df)
    print(f'Age range after clip: {df["age"].min()} – {df["age"].max()}')

    print('\n4. Build Text Column')
    df = build_text(df)
    print(f'Sample: {df["text"].iloc[0]}')

    print('\n5. Map Risk')
    df['risk'] = df['dx'].apply(map_risk)
    print(df['risk'].value_counts())

    print('\n6. Split Data')
    train_df, test_df = split_data(df)
    print(f'Train: {len(train_df)} | Test: {len(test_df)}')

    print('\n7. Save NLP Files')
    save_nlp_files(train_df, test_df)

    print('\n8. Leakage Check')
    overlap = set(train_df['lesion_id']) & set(test_df['lesion_id'])
    print(f'Shared lesions: {len(overlap)}')

    return train_df, test_df


In [32]:
train_df, test_df = run_pipeline_NLP('../data/HAM10000_metadata.csv')

1. Loading Data
Loaded: (10015, 7)

2. Handle Missing
Missing after: 0

3. Handle Outliers
Age range after clip: 2.5 – 85.0

4. Build Text Column
Sample: Dermatology case: 80 year old male patient. Lesion location: scalp. Diagnosis method: histo.

5. Map Risk
risk
low       8061
high      1113
medium     841
Name: count, dtype: int64

6. Split Data
Train: 8020 | Test: 1995

7. Save NLP Files
Saved: nlp_train.csv, nlp_test.csv

8. Leakage Check
Shared lesions: 0
